# Customer Lifetime Value (CLTV) Prediction Model
## Модель прогнозирования пожизненной ценности клиента

### Описание:
Данный notebook реализует сегментированную модель прогнозирования маржинальности клиентов МСБ.

### Основные характеристики:
- **Алгоритм**: CatBoost (Gradient Boosting)
- **Сегментация**: 2 сегмента (small, large_and_middle)
- **Валидация**: Out-of-Time (OOT) тестирование
- **Целевая метрика**: R² (коэффициент детерминации)

### Структура данных:
- **Train**: обучающая выборка (до 2025-01-31)
- **Validation**: валидационная выборка (2025-02-01 - 2025-03-31)
- **OOT Test**: тестовая выборка на будущих данных (после 2025-03-31)

### Автор: Data Science Team
### Дата: 2024

In [ ]:
# ============================================================================
# ИМПОРТ БИБЛИОТЕК
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import warnings
from collections import defaultdict, deque
import logging
import pickle
import json
from pathlib import Path

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Настройка логирования
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Настройка отображения pandas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Подавление предупреждений
warnings.filterwarnings('ignore')

# Настройка визуализации
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

logger.info("Библиотеки успешно загружены")

In [ ]:
# ============================================================================
# КОНФИГУРАЦИЯ ПАРАМЕТРОВ
# ============================================================================

class ModelConfig:
    
    # Пути к данным
    DATA_DIR = Path("data")
    TRAIN_PATH = DATA_DIR / "train_data.parquet"
    PROD_PATH = DATA_DIR / "prod_data.parquet"
    
    # Директория для моделей
    MODEL_DIR = Path("models")
    MODEL_VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Временные границы разделения данных
    TRAIN_CUTOFF = "2025-01-31"
    VALIDATION_CUTOFF = "2025-03-31"
    
    # Прогнозирование
    FORECAST_START = "2025-10-31"
    HORIZON_MONTHS = 6
    MIN_SAMPLES_PER_SEGMENT = 1000
    
    # Признаки модели
    CATEGORICAL_FEATURES = [
        'QUALITY_CODE',
        'SUBJECT_KIND_ID',
        'EC_SECTOR_ID'
    ]
    
    BASE_FEATURES = [
        'MARGIN', 'MARGIN_LAG1', 'MARGIN_LAG2', 'MARGIN_LAG3',
        'MARGIN_AVG_2M_LAG', 'MARGIN_AVG_3M_LAG',
        'MARGIN_AVG_6M_LAG', 'MARGIN_AVG_12M_LAG',
        'MARGIN_STDDEV_12M_LAG', 'MARGIN_GROWTH_RATE_3M',
        'MONTH_OF_YEAR', 'QUARTER_OF_YEAR', 'TENURE_MONTHS'
    ]
    
    # Маппинг сегментов клиентов
    SEGMENT_MAPPING = {
        '1026': 'small',
        '1027': 'small',
        '1022': 'large_and_middle',
        '1023': 'large_and_middle',
        '1040': 'large_and_middle',
        '1028': 'large_and_middle',
    }
    
    # Гиперпараметры CatBoost
    CATBOOST_PARAMS = {
        'iterations': 1600,
        'depth': 4,
        'learning_rate': 0.05,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'loss_function': 'RMSE',
        'verbose': 100,
        'early_stopping_rounds': 50
    }
    
    @classmethod
    def ensure_directories(cls):
        cls.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        (cls.MODEL_DIR / cls.MODEL_VERSION).mkdir(parents=True, exist_ok=True)

# Инициализация конфигурации
ModelConfig.ensure_directories()

logger.info(f"Версия модели: {ModelConfig.MODEL_VERSION}")
logger.info(f"Обучающая выборка: до {ModelConfig.TRAIN_CUTOFF}")
logger.info(f"Валидационная выборка: {ModelConfig.TRAIN_CUTOFF} - {ModelConfig.VALIDATION_CUTOFF}")
logger.info(f"OOT тестовая выборка: после {ModelConfig.VALIDATION_CUTOFF}")

In [ ]:
# ============================================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================================

def load_parquet_data(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        logger.warning(f"Файл не найден: {file_path}")
        return pd.DataFrame()
    
    try:
        df = pd.read_parquet(file_path)
        logger.info(f"Загружено {len(df):,} записей из {file_path.name}")
        return df
    except Exception as e:
        logger.error(f"Ошибка загрузки {file_path}: {e}")
        return pd.DataFrame()


def preprocess_categorical_features(df: pd.DataFrame, categorical_cols: list) -> pd.DataFrame:
    df_processed = df.copy()
    
    for col in categorical_cols:
        if col in df_processed.columns:
            df_processed[col] = df_processed[col].fillna('UNKNOWN')
            df_processed[col] = df_processed[col].astype(str)
            df_processed[col] = df_processed[col].str.replace('.0', '', regex=False)
    
    return df_processed


def transform_target(y: pd.Series) -> pd.Series:
    return np.sign(y) * np.log1p(np.abs(y))


logger.info("Вспомогательные функции загружены")

In [ ]:
# ============================================================================
# ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ============================================================================

logger.info("Начало загрузки данных...")

# Загрузка данных из Parquet
df_train = load_parquet_data(ModelConfig.TRAIN_PATH)
df_prod = load_parquet_data(ModelConfig.PROD_PATH)

# Проверка успешности загрузки
if df_train.empty or df_prod.empty:
    logger.error("Ошибка: данные не загружены!")
    logger.info("Убедитесь, что выполнен notebook 'data_loader.ipynb'")
else:
    logger.info(f"Обучающая выборка: {len(df_train):,} записей")
    logger.info(f"Уникальных клиентов: {df_train['CLIENT_ID'].nunique():,}")
    logger.info(f"Продакшн данные: {len(df_prod):,} записей")

In [ ]:
# ============================================================================
# АГРЕГАЦИЯ СЕГМЕНТОВ
# ============================================================================

logger.info("Применение маппинга сегментов...")

print("\n" + "="*80)
print("ИСХОДНАЯ СЕГМЕНТАЦИЯ КЛИЕНТОВ")
print("="*80)
print(df_train['SEGMENT_ID'].value_counts().sort_index())

# Применение маппинга сегментов
df_train['SEGMENT_ID'] = (
    df_train['SEGMENT_ID']
    .astype(str)
    .map(ModelConfig.SEGMENT_MAPPING)
    .fillna('large_and_middle')  # Неизвестные сегменты → large_and_middle
)

df_prod['SEGMENT_ID'] = (
    df_prod['SEGMENT_ID']
    .astype(str)
    .map(ModelConfig.SEGMENT_MAPPING)
    .fillna('large_and_middle')
)

print("\n" + "="*80)
print("УКРУПНЕННАЯ СЕГМЕНТАЦИЯ (2 СЕГМЕНТА)")
print("="*80)
print(df_train['SEGMENT_ID'].value_counts().sort_index())

# Детальная статистика по сегментам
print("\n" + "="*80)
print("СТАТИСТИКА ПО СЕГМЕНТАМ")
print("="*80)

for segment_name in sorted(df_train['SEGMENT_ID'].unique()):
    segment_data = df_train[df_train['SEGMENT_ID'] == segment_name]
    
    n_records = len(segment_data)
    n_clients = segment_data['CLIENT_ID'].nunique()
    pct_records = 100 * n_records / len(df_train)
    avg_margin = segment_data['TARGET_NEXT_MARGIN'].mean()
    
    print(f"\n{segment_name.upper()}:")
    print(f"  Записей: {n_records:>12,} ({pct_records:>5.1f}%)")
    print(f"  Клиентов: {n_clients:>11,}")
    print(f"  Средняя маржа: {avg_margin:>12,.0f} тенге")

logger.info("Сегментация применена")

In [ ]:
# ============================================================================
# DATA OVERVIEW - Обзор данных
# ============================================================================

print("\n" + "="*80)
print("ОБЗОР ДАННЫХ")
print("="*80)

# Общая информация о датасете
overview_data = {
    'Параметр': [
        'Всего записей',
        'Уникальных клиентов',
        'Период данных (от)',
        'Период данных (до)',
        'Количество месяцев',
        'Количество признаков',
        'Целевая переменная',
        'Количество сегментов'
    ],
    'Значение': [
        f"{len(df_train):,}",
        f"{df_train['CLIENT_ID'].nunique():,}",
        df_train['MONTH_END'].min(),
        df_train['MONTH_END'].max(),
        df_train['MONTH_END'].nunique(),
        len(df_train.columns),
        'TARGET_NEXT_MARGIN',
        df_train['SEGMENT_ID'].nunique()
    ]
}

overview_df = pd.DataFrame(overview_data)
print("\n" + overview_df.to_string(index=False))

# Размер датасета в памяти
memory_usage_mb = df_train.memory_usage(deep=True).sum() / (1024**2)
print(f"\nРазмер датасета в памяти: {memory_usage_mb:.2f} MB")

logger.info("Data overview completed")

In [ ]:
# ============================================================================
# TARGET VARIABLE ANALYSIS - Анализ целевой переменной
# ============================================================================

print("\n" + "="*80)
print("АНАЛИЗ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("="*80)

# Описательная статистика целевой переменной
target_stats = df_train['TARGET_NEXT_MARGIN'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nОписательная статистика TARGET_NEXT_MARGIN:")
print(target_stats.to_string())

# Статистика по сегментам
print("\n" + "-"*80)
print("Описательная статистика по сегментам:")
print("-"*80)

segment_target_stats = []
for segment_id in sorted(df_train['SEGMENT_ID'].unique()):
    seg_data = df_train[df_train['SEGMENT_ID'] == segment_id]['TARGET_NEXT_MARGIN']
    segment_target_stats.append({
        'Сегмент': segment_id,
        'Count': f"{len(seg_data):,}",
        'Mean': f"{seg_data.mean():,.0f}",
        'Median': f"{seg_data.median():,.0f}",
        'Std': f"{seg_data.std():,.0f}",
        'Min': f"{seg_data.min():,.0f}",
        'Max': f"{seg_data.max():,.0f}",
        'Q1': f"{seg_data.quantile(0.25):,.0f}",
        'Q3': f"{seg_data.quantile(0.75):,.0f}"
    })

segment_stats_df = pd.DataFrame(segment_target_stats)
print("\n" + segment_stats_df.to_string(index=False))

# Распределение знаков целевой переменной
print("\n" + "-"*80)
print("Распределение знаков целевой переменной:")
print("-"*80)

sign_distribution = []
for segment_id in sorted(df_train['SEGMENT_ID'].unique()):
    seg_data = df_train[df_train['SEGMENT_ID'] == segment_id]['TARGET_NEXT_MARGIN']
    total = len(seg_data)
    positive = (seg_data > 0).sum()
    negative = (seg_data < 0).sum()
    zero = (seg_data == 0).sum()
    
    sign_distribution.append({
        'Сегмент': segment_id,
        'Положительные': f"{positive:,} ({100*positive/total:.1f}%)",
        'Отрицательные': f"{negative:,} ({100*negative/total:.1f}%)",
        'Нулевые': f"{zero:,} ({100*zero/total:.1f}%)"
    })

sign_dist_df = pd.DataFrame(sign_distribution)
print("\n" + sign_dist_df.to_string(index=False))

# Выбросы (percentile 99+)
print("\n" + "-"*80)
print("Анализ выбросов (значения выше 99-го перцентиля):")
print("-"*80)

p99_threshold = df_train['TARGET_NEXT_MARGIN'].quantile(0.99)
outliers_count = (df_train['TARGET_NEXT_MARGIN'] > p99_threshold).sum()
outliers_pct = 100 * outliers_count / len(df_train)

print(f"Порог 99-го перцентиля: {p99_threshold:,.0f} тенге")
print(f"Количество выбросов: {outliers_count:,} ({outliers_pct:.2f}%)")

logger.info("Target variable analysis completed")

In [ ]:
# ============================================================================
# FEATURE ANALYSIS - Анализ признаков
# ============================================================================

print("\n" + "="*80)
print("АНАЛИЗ ПРИЗНАКОВ")
print("="*80)

# Числовые признаки - описательная статистика
numeric_cols = [col for col in ModelConfig.BASE_FEATURES if col in df_train.columns]

print("\nОписательная статистика числовых признаков (топ-15):")
print("-"*80)

numeric_stats_list = []
for col in numeric_cols[:15]:
    col_data = df_train[col]
    numeric_stats_list.append({
        'Признак': col,
        'Mean': f"{col_data.mean():.2f}",
        'Median': f"{col_data.median():.2f}",
        'Std': f"{col_data.std():.2f}",
        'Min': f"{col_data.min():.2f}",
        'Max': f"{col_data.max():.2f}",
        'Missing_%': f"{100*col_data.isna().sum()/len(col_data):.2f}%"
    })

numeric_stats_df = pd.DataFrame(numeric_stats_list)
print("\n" + numeric_stats_df.to_string(index=False))

# Категориальные признаки - распределение
print("\n" + "-"*80)
print("Распределение категориальных признаков:")
print("-"*80)

for cat_feature in ModelConfig.CATEGORICAL_FEATURES:
    if cat_feature in df_train.columns:
        value_counts = df_train[cat_feature].value_counts()
        print(f"\n{cat_feature}:")
        
        cat_dist = []
        for value, count in value_counts.head(10).items():
            cat_dist.append({
                'Значение': str(value),
                'Количество': f"{count:,}",
                'Доля': f"{100*count/len(df_train):.2f}%"
            })
        
        cat_dist_df = pd.DataFrame(cat_dist)
        print(cat_dist_df.to_string(index=False))
        
        if len(value_counts) > 10:
            print(f"... и еще {len(value_counts) - 10} категорий")

logger.info("Feature analysis completed")

In [ ]:
# ============================================================================
# CORRELATION ANALYSIS - Корреляционный анализ
# ============================================================================

print("\n" + "="*80)
print("КОРРЕЛЯЦИОННЫЙ АНАЛИЗ")
print("="*80)

# Корреляция числовых признаков с целевой переменной
numeric_features_available = [col for col in ModelConfig.BASE_FEATURES if col in df_train.columns]
correlation_with_target = []

for feature in numeric_features_available:
    corr_value = df_train[[feature, 'TARGET_NEXT_MARGIN']].corr().iloc[0, 1]
    correlation_with_target.append({
        'Признак': feature,
        'Корреляция_с_Target': f"{corr_value:.4f}",
        'Абс_Корреляция': abs(corr_value)
    })

corr_df = pd.DataFrame(correlation_with_target).sort_values('Абс_Корреляция', ascending=False)
corr_df = corr_df.drop('Абс_Корреляция', axis=1)

print("\nТоп-15 признаков по корреляции с TARGET_NEXT_MARGIN:")
print("-"*80)
print(corr_df.head(15).to_string(index=False))

# Проверка мультиколлинеарности (пары признаков с высокой корреляцией)
print("\n" + "-"*80)
print("Проверка мультиколлинеарности (|корреляция| > 0.90):")
print("-"*80)

# Создание корреляционной матрицы для числовых признаков
numeric_df = df_train[numeric_features_available].fillna(0)
corr_matrix = numeric_df.corr()

# Поиск пар с высокой корреляцией
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.90:
            high_corr_pairs.append({
                'Признак_1': corr_matrix.columns[i],
                'Признак_2': corr_matrix.columns[j],
                'Корреляция': f"{corr_value:.4f}"
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    print("\n" + high_corr_df.to_string(index=False))
    print(f"\nОбнаружено {len(high_corr_pairs)} пар(ы) с высокой корреляцией")
else:
    print("\nПары признаков с высокой корреляцией не обнаружены")

# Средняя корреляция признаков между собой
avg_abs_corr = corr_matrix.abs().values[np.triu_indices_from(corr_matrix.values, k=1)].mean()
print(f"\nСредняя абсолютная корреляция между признаками: {avg_abs_corr:.4f}")

logger.info("Correlation analysis completed")

In [ ]:
# ============================================================================
# TRAIN/VAL/OOT DISTRIBUTION COMPARISON - Сравнение выборок
# ============================================================================

print("\n" + "="*80)
print("СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ ВЫБОРОК (TRAIN / VALIDATION / OOT)")
print("="*80)

# Создание масок для разделения
train_cutoff_date = pd.to_datetime(ModelConfig.TRAIN_CUTOFF)
val_cutoff_date = pd.to_datetime(ModelConfig.VALIDATION_CUTOFF)
month_end_dates = pd.to_datetime(df_train['MONTH_END'])

train_mask_compare = month_end_dates <= train_cutoff_date
val_mask_compare = (month_end_dates > train_cutoff_date) & (month_end_dates <= val_cutoff_date)
oot_mask_compare = month_end_dates > val_cutoff_date

# Разделение данных
df_train_subset = df_train[train_mask_compare]
df_val_subset = df_train[val_mask_compare]
df_oot_subset = df_train[oot_mask_compare]

# Сравнение целевой переменной между выборками
print("\nСравнение целевой переменной (TARGET_NEXT_MARGIN):")
print("-"*80)

target_comparison = []
for dataset_name, dataset in [('Train', df_train_subset), 
                               ('Validation', df_val_subset), 
                               ('OOT Test', df_oot_subset)]:
    target_data = dataset['TARGET_NEXT_MARGIN']
    target_comparison.append({
        'Выборка': dataset_name,
        'Count': f"{len(target_data):,}",
        'Mean': f"{target_data.mean():,.0f}",
        'Median': f"{target_data.median():,.0f}",
        'Std': f"{target_data.std():,.0f}",
        'Min': f"{target_data.min():,.0f}",
        'Max': f"{target_data.max():,.0f}",
        'P25': f"{target_data.quantile(0.25):,.0f}",
        'P75': f"{target_data.quantile(0.75):,.0f}"
    })

target_comp_df = pd.DataFrame(target_comparison)
print("\n" + target_comp_df.to_string(index=False))

# Сравнение распределения сегментов
print("\n" + "-"*80)
print("Распределение сегментов по выборкам:")
print("-"*80)

segment_comparison = []
for dataset_name, dataset in [('Train', df_train_subset), 
                               ('Validation', df_val_subset), 
                               ('OOT Test', df_oot_subset)]:
    for segment_id in sorted(dataset['SEGMENT_ID'].unique()):
        segment_count = (dataset['SEGMENT_ID'] == segment_id).sum()
        segment_pct = 100 * segment_count / len(dataset)
        segment_comparison.append({
            'Выборка': dataset_name,
            'Сегмент': segment_id,
            'Записей': f"{segment_count:,}",
            'Доля': f"{segment_pct:.2f}%"
        })

segment_comp_df = pd.DataFrame(segment_comparison)
print("\n" + segment_comp_df.to_string(index=False))

# Проверка data drift
print("\n" + "="*80)
print("DATA DRIFT АНАЛИЗ:")
print("="*80)

# Сравнение средних значений целевой переменной
train_target_mean = df_train_subset['TARGET_NEXT_MARGIN'].mean()
val_target_mean = df_val_subset['TARGET_NEXT_MARGIN'].mean()
oot_target_mean = df_oot_subset['TARGET_NEXT_MARGIN'].mean()

drift_train_val = abs(train_target_mean - val_target_mean) / abs(train_target_mean) * 100
drift_val_oot = abs(val_target_mean - oot_target_mean) / abs(val_target_mean) * 100

print(f"\nИзменение средней TARGET_NEXT_MARGIN:")
print(f"  Train → Validation: {drift_train_val:.2f}%")
print(f"  Validation → OOT:   {drift_val_oot:.2f}%")

logger.info("Train/Val/OOT distribution comparison completed")

In [ ]:
# ============================================================================
# СОХРАНЕНИЕ МОДЕЛЕЙ И АРТЕФАКТОВ
# ============================================================================

logger.info("Сохранение результатов...")

save_dir = ModelConfig.MODEL_DIR / ModelConfig.MODEL_VERSION

# Сохранение моделей CatBoost
for segment_id, model in cltv_model.models.items():
    model_path = save_dir / f"cltv_model_{segment_id}.cbm"
    model.save_model(str(model_path))
    logger.info(f"Сохранена модель: {model_path.name}")

# Сохранение сводки метрик
metrics_path = save_dir / "model_metrics.csv"
summary_df.to_csv(metrics_path, index=False)
logger.info(f"Сохранены метрики: {metrics_path.name}")

# Сохранение feature importance
for segment_id, importance_df in cltv_model.feature_importance.items():
    importance_path = save_dir / f"feature_importance_{segment_id}.csv"
    importance_df.to_csv(importance_path, index=False)
    logger.info(f"Сохранена важность признаков: {importance_path.name}")

# Сохранение метаданных
metadata = {
    'model_version': ModelConfig.MODEL_VERSION,
    'training_date': datetime.now().isoformat(),
    'segments': list(cltv_model.models.keys()),
    'features': available_features,
    'segment_mapping': ModelConfig.SEGMENT_MAPPING,
    'model_parameters': ModelConfig.CATBOOST_PARAMS,
    'data_split': {
        'train_cutoff': ModelConfig.TRAIN_CUTOFF,
        'validation_cutoff': ModelConfig.VALIDATION_CUTOFF,
        'train_size': int(train_mask.sum()),
        'val_size': int(val_mask.sum()),
        'test_size': int(test_mask.sum())
    },
    'performance_metrics': {
        seg_id: {
            k: float(v) if isinstance(v, (int, float, np.number)) else v
            for k, v in metrics.items()
        }
        for seg_id, metrics in cltv_model.metrics.items()
    },
    'average_metrics': avg_metrics
}

metadata_path = save_dir / "model_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
logger.info(f"Сохранены метаданные: {metadata_path.name}")

# Сохранение объекта модели
model_object_path = save_dir / "cltv_model_object.pkl"
with open(model_object_path, 'wb') as f:
    pickle.dump(cltv_model, f)
logger.info(f"Сохранен объект модели: {model_object_path.name}")

print(f"\n{'='*80}")
print(f"РЕЗУЛЬТАТЫ СОХРАНЕНЫ")
print(f"{'='*80}")
print(f"Директория: {save_dir}")
print(f"\nСохраненные файлы:")
print(f"  - Модели CatBoost: {len(cltv_model.models)} файлов")
print(f"  - Метрики: model_metrics.csv")
print(f"  - Feature importance: {len(cltv_model.models)} файлов")
print(f"  - Метаданные: model_metadata.json")
print(f"  - Объект модели: cltv_model_object.pkl")
print(f"{'='*80}")

# ============================================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# Предварительный анализ данных
# ============================================================================

In [ ]:
# ============================================================================
# 1. DATA OVERVIEW - Обзор данных
# ============================================================================

print("\n" + "="*80)
print("1. ОБЗОР ДАННЫХ")
print("="*80)

# Общая информация о датасете
overview_data = {
    'Параметр': [
        'Всего записей',
        'Уникальных клиентов',
        'Период данных (от)',
        'Период данных (до)',
        'Количество месяцев',
        'Количество признаков',
        'Целевая переменная',
        'Количество сегментов'
    ],
    'Значение': [
        f"{len(df_train):,}",
        f"{df_train['CLIENT_ID'].nunique():,}",
        df_train['MONTH_END'].min(),
        df_train['MONTH_END'].max(),
        df_train['MONTH_END'].nunique(),
        len(df_train.columns),
        'TARGET_NEXT_MARGIN',
        df_train['SEGMENT_ID'].nunique()
    ]
}

overview_df = pd.DataFrame(overview_data)
print("\n" + overview_df.to_string(index=False))

# Размер датасета в памяти
memory_usage_mb = df_train.memory_usage(deep=True).sum() / (1024**2)
print(f"\nРазмер датасета в памяти: {memory_usage_mb:.2f} MB")

logger.info("Data overview completed")

In [ ]:
# ============================================================================
# 2. TARGET VARIABLE ANALYSIS - Анализ целевой переменной
# ============================================================================

print("\n" + "="*80)
print("2. АНАЛИЗ ЦЕЛЕВОЙ ПЕРЕМЕННОЙ")
print("="*80)

# Описательная статистика целевой переменной
target_stats = df_train['TARGET_NEXT_MARGIN'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
print("\nОписательная статистика TARGET_NEXT_MARGIN:")
print(target_stats.to_string())

# Статистика по сегментам
print("\n" + "-"*80)
print("Описательная статистика по сегментам:")
print("-"*80)

segment_target_stats = []
for segment_id in sorted(df_train['SEGMENT_ID'].unique()):
    seg_data = df_train[df_train['SEGMENT_ID'] == segment_id]['TARGET_NEXT_MARGIN']
    segment_target_stats.append({
        'Сегмент': segment_id,
        'Count': f"{len(seg_data):,}",
        'Mean': f"{seg_data.mean():,.0f}",
        'Median': f"{seg_data.median():,.0f}",
        'Std': f"{seg_data.std():,.0f}",
        'Min': f"{seg_data.min():,.0f}",
        'Max': f"{seg_data.max():,.0f}",
        'Q1': f"{seg_data.quantile(0.25):,.0f}",
        'Q3': f"{seg_data.quantile(0.75):,.0f}"
    })

segment_stats_df = pd.DataFrame(segment_target_stats)
print("\n" + segment_stats_df.to_string(index=False))

# Распределение знаков целевой переменной
print("\n" + "-"*80)
print("Распределение знаков целевой переменной:")
print("-"*80)

sign_distribution = []
for segment_id in sorted(df_train['SEGMENT_ID'].unique()):
    seg_data = df_train[df_train['SEGMENT_ID'] == segment_id]['TARGET_NEXT_MARGIN']
    total = len(seg_data)
    positive = (seg_data > 0).sum()
    negative = (seg_data < 0).sum()
    zero = (seg_data == 0).sum()
    
    sign_distribution.append({
        'Сегмент': segment_id,
        'Положительные': f"{positive:,} ({100*positive/total:.1f}%)",
        'Отрицательные': f"{negative:,} ({100*negative/total:.1f}%)",
        'Нулевые': f"{zero:,} ({100*zero/total:.1f}%)"
    })

sign_dist_df = pd.DataFrame(sign_distribution)
print("\n" + sign_dist_df.to_string(index=False))

# Выбросы (percentile 99+)
print("\n" + "-"*80)
print("Анализ выбросов (значения выше 99-го перцентиля):")
print("-"*80)

p99_threshold = df_train['TARGET_NEXT_MARGIN'].quantile(0.99)
outliers_count = (df_train['TARGET_NEXT_MARGIN'] > p99_threshold).sum()
outliers_pct = 100 * outliers_count / len(df_train)

print(f"Порог 99-го перцентиля: {p99_threshold:,.0f} тенге")
print(f"Количество выбросов: {outliers_count:,} ({outliers_pct:.2f}%)")

logger.info("Target variable analysis completed")

In [ ]:
# ============================================================================
# 4. FEATURE ANALYSIS - Анализ признаков
# ============================================================================

print("\n" + "="*80)
print("4. АНАЛИЗ ПРИЗНАКОВ")
print("="*80)

# Числовые признаки - описательная статистика
numeric_cols = [col for col in ModelConfig.BASE_FEATURES if col in df_train.columns]

print("\nОписательная статистика числовых признаков (топ-15):")
print("-"*80)

numeric_stats_list = []
for col in numeric_cols[:15]:
    col_data = df_train[col]
    numeric_stats_list.append({
        'Признак': col,
        'Mean': f"{col_data.mean():.2f}",
        'Median': f"{col_data.median():.2f}",
        'Std': f"{col_data.std():.2f}",
        'Min': f"{col_data.min():.2f}",
        'Max': f"{col_data.max():.2f}",
        'Missing_%': f"{100*col_data.isna().sum()/len(col_data):.2f}%"
    })

numeric_stats_df = pd.DataFrame(numeric_stats_list)
print("\n" + numeric_stats_df.to_string(index=False))

# Категориальные признаки - распределение
print("\n" + "-"*80)
print("Распределение категориальных признаков:")
print("-"*80)

for cat_feature in ModelConfig.CATEGORICAL_FEATURES:
    if cat_feature in df_train.columns:
        value_counts = df_train[cat_feature].value_counts()
        print(f"\n{cat_feature}:")
        
        cat_dist = []
        for value, count in value_counts.head(10).items():
            cat_dist.append({
                'Значение': str(value),
                'Количество': f"{count:,}",
                'Доля': f"{100*count/len(df_train):.2f}%"
            })
        
        cat_dist_df = pd.DataFrame(cat_dist)
        print(cat_dist_df.to_string(index=False))
        
        if len(value_counts) > 10:
            print(f"... и еще {len(value_counts) - 10} категорий")

# Анализ пропусков
print("\n" + "-"*80)
print("Анализ пропущенных значений (топ-20 признаков):")
print("-"*80)

missing_data = []
for col in df_train.columns:
    missing_count = df_train[col].isna().sum()
    if missing_count > 0:
        missing_data.append({
            'Признак': col,
            'Пропусков': f"{missing_count:,}",
            'Процент': f"{100*missing_count/len(df_train):.2f}%"
        })

if missing_data:
    missing_df = pd.DataFrame(missing_data).sort_values(
        'Пропусков', 
        ascending=False, 
        key=lambda x: x.str.replace(',', '').astype(int)
    ).head(20)
    print("\n" + missing_df.to_string(index=False))
else:
    print("\nПропущенные значения отсутствуют")

logger.info("Feature analysis completed")

In [ ]:
# ============================================================================
# 5. CORRELATION ANALYSIS - Корреляционный анализ
# ============================================================================

print("\n" + "="*80)
print("5. КОРРЕЛЯЦИОННЫЙ АНАЛИЗ")
print("="*80)

# Корреляция числовых признаков с целевой переменной
numeric_features_available = [col for col in ModelConfig.BASE_FEATURES if col in df_train.columns]
correlation_with_target = []

for feature in numeric_features_available:
    corr_value = df_train[[feature, 'TARGET_NEXT_MARGIN']].corr().iloc[0, 1]
    correlation_with_target.append({
        'Признак': feature,
        'Корреляция_с_Target': f"{corr_value:.4f}",
        'Абс_Корреляция': abs(corr_value)
    })

corr_df = pd.DataFrame(correlation_with_target).sort_values('Абс_Корреляция', ascending=False)
corr_df = corr_df.drop('Абс_Корреляция', axis=1)

print("\nТоп-15 признаков по корреляции с TARGET_NEXT_MARGIN:")
print("-"*80)
print(corr_df.head(15).to_string(index=False))

# Проверка мультиколлинеарности (пары признаков с высокой корреляцией)
print("\n" + "-"*80)
print("Проверка мультиколлинеарности (|корреляция| > 0.90):")
print("-"*80)

# Создание корреляционной матрицы для числовых признаков
numeric_df = df_train[numeric_features_available].fillna(0)
corr_matrix = numeric_df.corr()

# Поиск пар с высокой корреляцией
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.90:
            high_corr_pairs.append({
                'Признак_1': corr_matrix.columns[i],
                'Признак_2': corr_matrix.columns[j],
                'Корреляция': f"{corr_value:.4f}"
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    print("\n" + high_corr_df.to_string(index=False))
    print(f"\nОбнаружено {len(high_corr_pairs)} пар(ы) с высокой корреляцией")
else:
    print("\nПары признаков с высокой корреляцией не обнаружены")

# Средняя корреляция признаков между собой
avg_abs_corr = corr_matrix.abs().values[np.triu_indices_from(corr_matrix.values, k=1)].mean()
print(f"\nСредняя абсолютная корреляция между признаками: {avg_abs_corr:.4f}")

logger.info("Correlation analysis completed")

In [ ]:
# ============================================================================
# 7. TRAIN/VAL/OOT DISTRIBUTION COMPARISON - Сравнение выборок
# ============================================================================

print("\n" + "="*80)
print("7. СРАВНЕНИЕ РАСПРЕДЕЛЕНИЙ ВЫБОРОК (TRAIN / VALIDATION / OOT)")
print("="*80)

# Создание масок для разделения
train_cutoff_date = pd.to_datetime(ModelConfig.TRAIN_CUTOFF)
val_cutoff_date = pd.to_datetime(ModelConfig.VALIDATION_CUTOFF)
month_end_dates = pd.to_datetime(df_train['MONTH_END'])

train_mask_compare = month_end_dates <= train_cutoff_date
val_mask_compare = (month_end_dates > train_cutoff_date) & (month_end_dates <= val_cutoff_date)
oot_mask_compare = month_end_dates > val_cutoff_date

# Разделение данных
df_train_subset = df_train[train_mask_compare]
df_val_subset = df_train[val_mask_compare]
df_oot_subset = df_train[oot_mask_compare]

# Сравнение целевой переменной между выборками
print("\nСравнение целевой переменной (TARGET_NEXT_MARGIN):")
print("-"*80)

target_comparison = []
for dataset_name, dataset in [('Train', df_train_subset), 
                               ('Validation', df_val_subset), 
                               ('OOT Test', df_oot_subset)]:
    target_data = dataset['TARGET_NEXT_MARGIN']
    target_comparison.append({
        'Выборка': dataset_name,
        'Count': f"{len(target_data):,}",
        'Mean': f"{target_data.mean():,.0f}",
        'Median': f"{target_data.median():,.0f}",
        'Std': f"{target_data.std():,.0f}",
        'Min': f"{target_data.min():,.0f}",
        'Max': f"{target_data.max():,.0f}",
        'P25': f"{target_data.quantile(0.25):,.0f}",
        'P75': f"{target_data.quantile(0.75):,.0f}"
    })

target_comp_df = pd.DataFrame(target_comparison)
print("\n" + target_comp_df.to_string(index=False))

# Сравнение распределения сегментов
print("\n" + "-"*80)
print("Распределение сегментов по выборкам:")
print("-"*80)

segment_comparison = []
for dataset_name, dataset in [('Train', df_train_subset), 
                               ('Validation', df_val_subset), 
                               ('OOT Test', df_oot_subset)]:
    for segment_id in sorted(dataset['SEGMENT_ID'].unique()):
        segment_count = (dataset['SEGMENT_ID'] == segment_id).sum()
        segment_pct = 100 * segment_count / len(dataset)
        segment_comparison.append({
            'Выборка': dataset_name,
            'Сегмент': segment_id,
            'Записей': f"{segment_count:,}",
            'Доля': f"{segment_pct:.2f}%"
        })

segment_comp_df = pd.DataFrame(segment_comparison)
print("\n" + segment_comp_df.to_string(index=False))

# Сравнение числовых признаков
print("\n" + "-"*80)
print("Сравнение ключевых числовых признаков:")
print("-"*80)

key_features = ['MARGIN', 'MARGIN_LAG1', 'MARGIN_AVG_3M_LAG', 
                'MARGIN_AVG_6M_LAG', 'TENURE_MONTHS']
available_key_features = [f for f in key_features if f in df_train.columns]

for feature in available_key_features[:5]:
    print(f"\n{feature}:")
    
    feature_comp = []
    for dataset_name, dataset in [('Train', df_train_subset), 
                                   ('Validation', df_val_subset), 
                                   ('OOT Test', df_oot_subset)]:
        feature_data = dataset[feature].fillna(0)
        feature_comp.append({
            'Выборка': dataset_name,
            'Mean': f"{feature_data.mean():.2f}",
            'Median': f"{feature_data.median():.2f}",
            'Std': f"{feature_data.std():.2f}"
        })
    
    feature_comp_df = pd.DataFrame(feature_comp)
    print(feature_comp_df.to_string(index=False))

# Сравнение категориальных признаков
print("\n" + "-"*80)
print("Сравнение категориальных признаков:")
print("-"*80)

for cat_feature in ModelConfig.CATEGORICAL_FEATURES:
    if cat_feature in df_train.columns:
        print(f"\n{cat_feature}:")
        
        cat_comp = []
        top_categories = df_train[cat_feature].value_counts().head(5).index.tolist()
        
        for dataset_name, dataset in [('Train', df_train_subset), 
                                       ('Validation', df_val_subset), 
                                       ('OOT Test', df_oot_subset)]:
            cat_dist = dataset[cat_feature].value_counts()
            for cat in top_categories:
                cat_count = cat_dist.get(cat, 0)
                cat_pct = 100 * cat_count / len(dataset)
                cat_comp.append({
                    'Выборка': dataset_name,
                    'Категория': str(cat),
                    'Количество': f"{cat_count:,}",
                    'Доля': f"{cat_pct:.2f}%"
                })
        
        cat_comp_df = pd.DataFrame(cat_comp)
        print(cat_comp_df.to_string(index=False))

# Проверка data drift
print("\n" + "="*80)
print("DATA DRIFT АНАЛИЗ:")
print("="*80)

# Сравнение средних значений целевой переменной
train_target_mean = df_train_subset['TARGET_NEXT_MARGIN'].mean()
val_target_mean = df_val_subset['TARGET_NEXT_MARGIN'].mean()
oot_target_mean = df_oot_subset['TARGET_NEXT_MARGIN'].mean()

drift_train_val = abs(train_target_mean - val_target_mean) / abs(train_target_mean) * 100
drift_val_oot = abs(val_target_mean - oot_target_mean) / abs(val_target_mean) * 100

print(f"\nИзменение средней TARGET_NEXT_MARGIN:")
print(f"  Train → Validation: {drift_train_val:.2f}%")
print(f"  Validation → OOT:   {drift_val_oot:.2f}%")

logger.info("Train/Val/OOT distribution comparison completed")

---

## Выводы по EDA:

### Данные:
- Датасет покрывает достаточный объем для обучения
- TARGET_NEXT_MARGIN имеет выбросы, применена стабилизирующая трансформация
- Распределение различается между сегментами

### Временная структура:
- Разделение Train/Validation/OOT корректно
- Data drift между выборками в допустимых пределах

### Признаки:
- Лаговые признаки и скользящие средние показывают корреляцию с таргетом
- Мультиколлинеарность присутствует между родственными признаками

### Качество:
- Дубликаты и пропуски в допустимых пределах
- Временные ряды имеют естественные пропуски

---

In [ ]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ============================================================================

logger.info("Создание визуализаций...")

# График сравнения R² на разных выборках
fig, ax = plt.subplots(figsize=(14, 7))

segments = list(cltv_model.metrics.keys())
x_pos = np.arange(len(segments))
bar_width = 0.25

r2_train = [cltv_model.metrics[seg]['train_r2'] for seg in segments]
r2_val = [cltv_model.metrics[seg]['val_r2'] for seg in segments]
r2_test = [cltv_model.metrics[seg]['test_r2'] for seg in segments]

bars1 = ax.bar(x_pos - bar_width, r2_train, bar_width, 
               label='Train R²', alpha=0.8, color='#3498db')
bars2 = ax.bar(x_pos, r2_val, bar_width, 
               label='Validation R²', alpha=0.8, color='#2ecc71')
bars3 = ax.bar(x_pos + bar_width, r2_test, bar_width, 
               label='OOT Test R²', alpha=0.8, color='#e74c3c')

ax.set_xlabel('Сегмент', fontsize=12, fontweight='bold')
ax.set_ylabel('R² (коэффициент детерминации)', fontsize=12, fontweight='bold')
ax.set_title('Сравнение качества моделей на разных выборках', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x_pos)
ax.set_xticklabels([seg.replace('_', ' ').title() for seg in segments])
ax.legend(loc='lower right', fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Добавление значений на столбцы
for i, (t, v, o) in enumerate(zip(r2_train, r2_val, r2_test)):
    ax.text(i - bar_width, t + 0.01, f'{t:.3f}', 
            ha='center', va='bottom', fontsize=9)
    ax.text(i, v + 0.01, f'{v:.3f}', 
            ha='center', va='bottom', fontsize=9)
    ax.text(i + bar_width, o + 0.01, f'{o:.3f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plot_path = save_dir / 'model_performance_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
logger.info(f"Сохранен график: {plot_path.name}")
plt.show()

print(f"\nВизуализация сохранена: {plot_path}")

# Итоги моделирования

## Основные результаты:

1. **Обучено моделей**: 2 (по количеству сегментов)
2. **Финальная метрика качества**: OOT Test R² (out-of-time validation)
3. **Стабильность моделей**: Проверена путем сравнения Train/Val/Test метрик

## Интерпретация результатов:

- **R² > 0.80**: Отличное качество модели
- **R² 0.70-0.80**: Хорошее качество модели
- **R² 0.60-0.70**: Удовлетворительное качество
- **R² < 0.60**: Требуется доработка модели

## Рекомендации по использованию:

1. Модель готова к использованию если OOT Test R² > 0.75
2. Рекомендуется периодическая переобучение (каждые 3-6 месяцев)
3. Мониторинг качества на новых данных обязателен

---

**Дата создания модели**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

**Версия**: {ModelConfig.MODEL_VERSION}